# Sort from a handful

Show it a few examples per label and it sorts the rest. When it isn't sure, it puts the item in "needs a human" instead of guessing. No training run, no API key, runs on CPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fuellabs-ai/cookbooks/blob/main/sort-from-a-handful/walkthrough.ipynb)

This notebook runs the recipe end to end: split 200 real GitHub issues into 50 dev and 150 test issues, strip issue-template boilerplate, pick the margin on dev only, report on test only, sort with a real handful and with a fictional one, fix mistakes with the corrections loop, and sort two fictional shared inboxes.

The model is HobbyLM-Embed 140M through Sentence Transformers with `trust_remote_code=True`. Classification uses plain text with no `prompt_name`. Vectors are unit-norm.

## Setup

The cell below sets the repository and branch, and falls back to cloning them when the notebook is not running inside a checkout.

In [1]:
REPO = "fuellabs-ai/cookbooks"
BRANCH = "main"
FOLDER = "sort-from-a-handful"

import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

if Path("sort_handful.py").exists():
    print("running from a local checkout")
else:
    if not Path(FOLDER, "sort_handful.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH,
             f"https://github.com/{REPO}", FOLDER],
            check=True,
        )
    os.chdir(FOLDER)
print("working folder:", Path.cwd())

running from a local checkout
working folder: /Users/prajhan/refuel/hobbylm-embed-cookbook-repo/sort-from-a-handful


## Install and load the model

Nothing here needs a GPU. On a laptop CPU the model loads in a few seconds.

In [2]:
try:
    import sentence_transformers
    print("sentence-transformers already installed:", sentence_transformers.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    print("installed sentence-transformers")

import time
import numpy as np
from sort_handful import load_model

start = time.perf_counter()
model = load_model()  # harims95/hobbylm-embed-140m, trust_remote_code=True
print(f"model loaded in {time.perf_counter() - start:.1f} s")

sample = model.encode(
    ["fit() crashes with a segmentation fault when n_jobs=-1",
     "How do I get the feature names out of a fitted pipeline?"],
    convert_to_numpy=True,
)
print("dim:", sample.shape[1], "norms:", np.linalg.norm(sample, axis=1).round(3))

W1123 11:44:40.505000 40166 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


sentence-transformers already installed: 6.3.0


model loaded in 1.7 s


dim: 512 norms: [1. 1.]


## 200 real GitHub issues, split once into 50 dev and 150 test

`examples/github_issues/` holds 200 closed issues from [scikit-learn/scikit-learn](https://github.com/scikit-learn/scikit-learn) (BSD-3 licence), 50 per label, with the maintainers' own labels: `bug`, `documentation`, `question`, `new feature`. They were sampled from the public GitHub API on 21 November 2026: closed issues with the label, no pull requests, a body of at least 80 characters, 50 picked per label with seed 7.

The split is fixed once with seed 7 and stratified: 50 dev issues (13 bug, 13 documentation, 12 question, 12 new feature) in `dev.csv`, 150 test issues in `test.csv`. The margin below is picked on dev only. Every number after that is reported on the same 150 test issues.

## What stripping does

`--strip-templates` removes issue-template boilerplate before embedding: markdown headings, template prompts like "Describe the workflow you want to enable" and "Steps/Code to Reproduce", HTML comments, placeholder answers, and empty checklist lines. The title and the person's own words stay.

In [3]:
from sort_handful import strip_issue_text
import csv, random

rows = list(csv.DictReader(open("examples/github_issues/items.csv", encoding="utf-8")))
rng = random.Random(3)
shown = 0
for row in rng.sample(rows, 30):
    if "Describe the" in row["text"] and shown < 2:
        shown += 1
        print("BEFORE:", row["text"][:180])
        print("AFTER :", strip_issue_text(row["text"])[:180])
        print("-" * 60)
for row in rows:
    if row["number"] == "35285":
        print("BEFORE:", row["text"][:180])
        print("AFTER :", strip_issue_text(row["text"])[:180])

BEFORE: [FEA] Add Ambiguity Range metric for safety-critical classification > [!WARNING] > This issue is not yet ready for a PR. If you are interested in contributing to scikit-learn, plea
AFTER : [FEA] Add Ambiguity Range metric for safety-critical classification > Practitioners in safety-critical fields—such as medical diagnostics, structural engineering, or high-stakes fi
------------------------------------------------------------
BEFORE: DOC: n_jobs parameter in LogisticRegression missing inline deprecation notice > [!WARNING] > This issue is not yet ready for a PR. If you are interested in contributing to scikit-l
AFTER : DOC: n_jobs parameter in LogisticRegression missing inline deprecation notice > Add the deprecation notice inline in the `n_jobs` parameter description block in `sklearn/linear_mod
------------------------------------------------------------
BEFORE: HTML repr: keyboard focus is invisible on the expand/collapse toggles (WCAG 2.4.7) > [!WARNING] > This issue is no

## Sort the 150 test issues from a real handful

The handful is 10 examples per label sampled from the dev issues (seed 7), so no test issue is ever used as an example. The margin comes from `margin.json`, picked on dev by leave-one-out cross-validation: each dev item was scored against a handful drawn from the other dev items, and the sweep ran on those scores.

In [4]:
result = subprocess.run(
    [sys.executable, "sort_handful.py", "examples/github_issues/test.csv",
     "--labels", "examples/github_issues/labels.yaml",
     "--strip-templates",
     "--examples-from", "examples/github_issues/dev.csv", "--per-label", "10"],
    capture_output=True, text=True,
)
if result.returncode:
    print(result.stderr)
lines = [line for line in result.stdout.splitlines() if line.strip()]
print("\n".join(lines[:12]))
print("... output trimmed; every deferred item is in report.html")

examples: 10 per label sampled from examples/github_issues/dev.csv (seed 7)
wrote examples/github_issues/sorted.csv
wrote examples/github_issues/report.html
items: 150  margin: 0.010  method: nearest-centroid
sorted: 73  sent to a person: 77
wrongly sorted: 30
  radius_neighbors with array radius works with n_jobs=1 but raises ValueError for n_jobs>1 When `radius` is passed as a p... -> question (expected bug, score 0.838, margin 0.036)
  Automatic main lock-file update fails for debian-32 build I can reproduce locally, no clue where this is coming from: ``... -> documentation (expected bug, score 0.743, margin 0.015)
  HTML repr: keyboard focus is invisible on the expand/collapse toggles (WCAG 2.4.7) > Hi! I'm Eya, a full-stack developer... -> documentation (expected bug, score 0.720, margin 0.010)
  `UnionFind.fast_find`'s path compression never touches the queried node — quadratic linkage labelling `UnionFind.fast_fi... -> question (expected bug, score 0.815, margin 0.027)
  Feature

## What it labelled, what it deferred

At the margin picked on dev it labels 73 of 150 and sends 77 to a person, with 30 wrongly sorted among the labelled ones. Half of the test set still reaches a person, and one in three labelled items carries the wrong label: scikit-learn's four labels differ mostly by intent, and the wording alone does not separate them.

In [5]:
import pandas as pd

sorted_csv = pd.read_csv("examples/github_issues/sorted.csv")
wrongly = sorted_csv[(sorted_csv.needs_human == False) & (sorted_csv.label != sorted_csv.expected)]
deferred = sorted_csv[sorted_csv.needs_human == True]
print(f"labelled: {len(sorted_csv) - len(deferred)}  wrongly sorted: {len(wrongly)}  sent to a person: {len(deferred)}")
print()
print("wrongly sorted:")
for _, row in wrongly.iterrows():
    print(f"  got {row.label} (expected {row.expected}) | {row.text[:80]}...")
print()
print("nearest calls among the deferred:")
print(deferred.groupby("label").size().sort_values(ascending=False).head(6).to_string())

labelled: 73  wrongly sorted: 30  sent to a person: 77

wrongly sorted:
  got question (expected bug) | radius_neighbors with array radius works with n_jobs=1 but raises ValueError for...
  got documentation (expected bug) | Automatic main lock-file update fails for debian-32 build I can reproduce locall...
  got documentation (expected bug) | HTML repr: keyboard focus is invisible on the expand/collapse toggles (WCAG 2.4....
  got question (expected bug) | `UnionFind.fast_find`'s path compression never touches the queried node — quadra...
  got question (expected bug) | FeatureAgglomeration.inverse_transform fails on list/DataFrame input, including ...
  got documentation (expected bug) | Keyword-Only arguments in `train_test_split` > It looks like in https://github.c...
  got question (expected bug) | Free-threaded failure in `gaussian_process/tests/test_gpc.py::test_multi_class_n...
  got question (expected documentation) | DOC: add explanations about criterion="friedman_mse" in dec

## Corrections loop

Move an item to the right label, put it in a CSV with `text` and `label` columns, and pass it with `--corrections`. It is appended as a new example, the centroids are rebuilt, and the run repeats. The three corrections here are dev items the sorter mislabelled during the dev sweep; nothing from the test set is used.

In [6]:
result = subprocess.run(
    [sys.executable, "sort_handful.py", "examples/github_issues/test.csv",
     "--labels", "examples/github_issues/labels.yaml",
     "--strip-templates",
     "--examples-from", "examples/github_issues/dev.csv", "--per-label", "10",
     "--corrections", "examples/github_issues/corrections.csv"],
    capture_output=True, text=True,
)
lines = [line for line in result.stdout.splitlines() if line.strip()]
print("\n".join(lines[:10]))

appending 3 corrections as new examples
examples: 10 per label sampled from examples/github_issues/dev.csv (seed 7)
wrote examples/github_issues/sorted.csv
wrote examples/github_issues/report.html
items: 150  margin: 0.010  method: nearest-centroid
sorted: 80  sent to a person: 70
wrongly sorted: 35
  radius_neighbors with array radius works with n_jobs=1 but raises ValueError for n_jobs>1 When `radius` is passed as a p... -> question (expected bug, score 0.832, margin 0.030)
  `UnionFind.fast_find`'s path compression never touches the queried node — quadratic linkage labelling `UnionFind.fast_fi... -> question (expected bug, score 0.813, margin 0.025)
  FeatureAgglomeration.inverse_transform fails on list/DataFrame input, including its own set_output result > I work in AI... -> question (expected bug, score 0.842, margin 0.019)


## Calibrate the margin on dev

`calibrate.py` sweeps margins on the dev scores, prints wrongly-sorted versus sent-to-human at each, and recommends the smallest margin that keeps wrongly-sorted items at or under one in ten labelled items. On this dev set no margin manages that, so it says so and takes the margin with the lowest wrongly-sorted rate instead. It also sorts with 5, 10 and 25 examples per label from dev, reporting every row on the same 150 test issues.

In [7]:
result = subprocess.run(
    [sys.executable, "calibrate.py", "examples/github_issues",
     "--strip-templates",
     "--examples-from", "examples/github_issues/dev.csv", "--per-label", "10",
     "--test", "examples/github_issues/test.csv"],
    capture_output=True, text=True,
)
lines = [line for line in result.stdout.splitlines() if line.strip()]
keep = []
skipping = False
for line in lines:
    # show the sweep and the summary, trim the long per-item lists
    if line.startswith("  wrongly-sorted") or line.startswith("  sent-to-human"):
        skipping = True
        continue
    if skipping and not line.startswith("  "):
        skipping = False
    if not skipping:
        keep.append(line)
print("\n".join(keep[:50]))

=== github_issues ===
examples: 10 per label sampled from examples/github_issues/dev.csv (seed 7)
items: 50 (13 bug, 13 documentation, 12 question, 12 new feature)
margin  labelled  wrongly-sorted  sent-to-human
 0.00       50    23/50           0/50
 0.02       18    7/50           32/50
 0.04        2    0/50           48/50
 0.06        0    0/50           50/50
 0.08        0    0/50           50/50
 0.10        0    0/50           50/50
 0.12        0    0/50           50/50
 0.14        0    0/50           50/50
 0.16        0    0/50           50/50
 0.18        0    0/50           50/50
 0.20        0    0/50           50/50
 0.22        0    0/50           50/50
 0.24        0    0/50           50/50
 0.26        0    0/50           50/50
 0.28        0    0/50           50/50
 0.30        0    0/50           50/50
no margin in the sweep kept wrongly-sorted items to one in ten labelled items on dev. The lowest rate is at margin 0.010 (labelled 26/50, wrongly-sorted 8, sent-to-

## The fictional inboxes

Two fictional shared inboxes, written once and not tuned afterwards:

- `examples/careers_inbox`: a careers@ address with referral, direct application, agency CV, interview scheduling and spam. `referral` carries `never_hide: true`, so a referral always reaches a person even when the sorter is sure.
- `examples/school_admissions`: an admissions@ address with fees, documents, visit request, transfer and complaint. `complaint` carries `never_hide: true`.

Accuracy on fictional data mostly measures how clearly the example texts were written.

In [8]:
for folder in ["careers_inbox", "school_admissions"]:
    result = subprocess.run(
        [sys.executable, "sort_handful.py", f"examples/{folder}/inbox.csv",
         "--labels", f"examples/{folder}/labels.yaml"],
        capture_output=True, text=True,
    )
    print("=" * 20, folder, "=" * 20)
    print(result.stdout.strip())

==================== careers_inbox ====================
wrote examples/careers_inbox/sorted.csv
wrote examples/careers_inbox/report.html

items: 30  margin: 0.020  method: nearest-centroid
sorted: 20  sent to a person: 10
wrongly sorted: 2
  My old flatmate Tomasz built the flight controller at AeroWorks and I think he would be great for your firmware opening.... -> direct application (expected referral, score 0.850, margin 0.031)
  I applied for the firmware role two weeks ago and have not heard back. Could you confirm my application arrived and tell... -> direct application (expected interview scheduling, score 0.834, margin 0.020)
sent to a person:
  Hi, this is Marco from the perception team. My former colleague Ilse reviewed our SLAM stack last year and is now lookin... -> referral or direct application (margin 0.063)
  Forwarding on behalf of my brother in law, who has ten years in warehouse automation. He is too modest to write this him... -> direct application or referral (marg

==================== school_admissions ====================
wrote examples/school_admissions/sorted.csv
wrote examples/school_admissions/report.html

items: 30  margin: 0.000  method: nearest-centroid
sorted: 24  sent to a person: 6
wrongly sorted: 0
sent to a person:
  I am writing to complain that the after school club billed us for sessions in October that never ran and nobody has repl... -> complaint or fees (margin 0.036)
  This is the third time this month the school bus has been over 20 minutes late. I need someone to look into it properly. -> complaint or visit request (margin 0.103)
  A teacher told my daughter she was hopeless at maths in front of the class. I want to speak to someone senior about how ... -> complaint or visit request (margin 0.140)
  We were charged twice for the winter concert tickets. The box office says it is the admissions office's problem. Please ... -> complaint or fees (margin 0.033)
  I am not happy that our complaint about the lost uniform was close

## Report

Each run writes `sorted.csv` (text, label, score, margin, needs_human, plus the expected label when the input has one) and `report.html` with the per-label table, the deferred items with their top 2 labels, and every wrong sort. Open the HTML in a browser.

## Limits and where to go next

It sorts by topic and wording, not intent. Labels that differ mostly in intent or tone (a new lead vs a vendor pitch, an agency CV vs spam) will blur, and the margin can be wrong both ways, so set it on your own items and keep a person on anything urgent or sensitive. English only. It labels; it never acts.

The recipe pairs with the What people keep ask for cookbook: first see what people say, name the groups, then sort everything that comes next into them.

## Files

- `sort_handful.py`: the CLI. `python sort_handful.py items.csv --labels labels.yaml`, with `--margin`, `--method`, `--k`, `--corrections`, `--strip-templates`, `--examples-from`.
- `calibrate.py`: the margin sweep on dev and the 5/10/25 table on test.
- `labels.yaml`: the fictional handful, with optional `never_hide: true`.
- `examples/github_issues/dev.csv`, `test.csv`: the fixed 50/150 split (seed 7).
- `walkthrough.ipynb`: the executed walkthrough, with an Open in Colab badge.
- `.github/workflows/label-issues.example.yml`: a GitHub Action that labels new issues on CPU with no secrets beyond the default `GITHUB_TOKEN`.